In [3]:
from pathlib import Path
from openai import OpenAI
import chromadb

# ---------- Ollama setup (free, local) ----------
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama_key")

MODEL = "llama3.2"               # chat model
EMB_MODEL = "nomic-embed-text"   # embedding model (768 dims)


def ask(messages, temperature=0):
    try:
        r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature)
        return (r.choices[0].message.content or "").strip()
    except Exception as e:
        return f"ERROR: {e}"


def embed(texts, kind="document"):
    # nomic ko prefix chahiye, warna retrieval quality girti hai
    prefix = "search_document: " if kind == "document" else "search_query: "
    texts = [prefix + t for t in texts]
    r = client.embeddings.create(model=EMB_MODEL, input=texts)
    return [item.embedding for item in r.data]


# ---------- Documents ----------
MANAGER_ONLY = ["salary_bands.txt"]

def load_documents(folder):
    docs = []
    for path in sorted(Path(folder).glob("*.txt")):
        docs.append({"source": path.name, "text": path.read_text(encoding="utf-8")})
    return docs

def chunk_by_paragraph(text):
    paragraphs = []
    for p in text.split("\n\n"):
        p = p.strip()
        if len(p) > 30:
            paragraphs.append(p)
    return paragraphs

def chunk_documents(docs):
    chunks = []
    for doc in docs:
        for i, text in enumerate(chunk_by_paragraph(doc["text"])):
            chunks.append({
                "id": f"{doc['source']}-{i}",
                "source": doc["source"],
                "access": "manager" if doc["source"] in MANAGER_ONLY else "all",
                "text": text,
            })
    return chunks

chunks = chunk_documents(load_documents("company_docs"))


# ---------- Vector DB (NAYA folder + NAYA naam) ----------
db = chromadb.PersistentClient(path="company_db_nomic")
store = db.get_or_create_collection(
    name="company_docs_nomic",
    metadata={"hnsw:space": "cosine"},
)

if store.count() == 0:
    ids, documents, metadatas = [], [], []
    for c in chunks:
        ids.append(c["id"])
        documents.append(c["text"])
        metadatas.append({"source": c["source"], "access": c["access"]})

    vecs = embed(documents, kind="document")
    print("embedding dimension:", len(vecs[0]))      # 768 aana chahiye
    store.add(ids=ids, documents=documents, metadatas=metadatas, embeddings=vecs)

print(store.count(), "chunks in the database")


def retrieve(question, n=3):
    r = store.query(query_embeddings=embed([question], kind="query"), n_results=n)
    texts = r["documents"][0]
    sources = r["metadatas"][0]
    distances = r["distances"][0]

    found = []
    for i in range(len(texts)):
        found.append({
            "text": texts[i],
            "source": sources[i]["source"],
            "distance": distances[i],
        })
    return found

22 chunks in the database


In [4]:
# ---------- RAG ----------
RAG_SYSTEM = """You are the HR support bot of a company.
Answer the employee's question using ONLY the context below.
If the answer is not in the context, reply exactly: "I don't know, this is not in the company documents."
Do not use outside knowledge, even if you know the answer.
Be friendly. Reply in at most 3 short sentences.
End the answer with the source file in square brackets, like [leave_policy.txt]."""

MAX_DISTANCE = 0.55    # run_tests() ki distances dekh ke tune karna

def rag(question):
    found = retrieve(question)
    found = [f for f in found if f["distance"] <= MAX_DISTANCE]

    if not found:
        return "I don't know, this is not in the company documents."

    pieces = [f"[{f['source']}]\n{f['text']}" for f in found]
    context = "\n\n".join(pieces)

    messages = [
        {"role": "system", "content": RAG_SYSTEM},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    return ask(messages)


# ---------- Tests ----------
tests = [
    ("How many days of paid leave do I get?",                 "20 days [leave_policy.txt]"),
    ("I have a fever and cannot come to work. What do I do?", "10 sick days, doctor note after 2 days in a row [leave_policy.txt]"),
    ("Can I install a game on my laptop?",                    "No, ask the IT team first [laptop_rules.txt]"),
    ("Does the company pay for my home internet?",            "Yes, Rs 3000 each month [remote_work.txt]"),
    ("Can I bring my dog to the office?",                     "I don't know (not in the documents)"),
    ("What is the capital of Pakistan?",                      "I don't know (not about the company)"),
]

def run_tests():
    for question, expected in tests:
        print("Q:       ", question)
        print("Expected:", expected)
        print("Bot:     ", rag(question))
        best = retrieve(question, n=1)[0]
        print(f"(best distance: {best['distance']:.3f} from {best['source']})\n")

run_tests()

Q:        How many days of paid leave do I get?
Expected: 20 days [leave_policy.txt]
Bot:      You get 20 days of paid leave in one year. This amount does not change regardless of the year. [leave_policy.txt]
(best distance: 0.151 from leave_policy.txt)

Q:        I have a fever and cannot come to work. What do I do?
Expected: 10 sick days, doctor note after 2 days in a row [leave_policy.txt]
Bot:      You should send a request to your manager to take sick leave as soon as possible, and they will guide you on the next steps. Since you have a fever, you'll need to wait until you've had 2 sick days in a row to show a doctor's note. Make sure to send your request at least 5 days before your intended absence.

[leave_policy.txt]
(best distance: 0.355 from leave_policy.txt)

Q:        Can I install a game on my laptop?
Expected: No, ask the IT team first [laptop_rules.txt]
Bot:      You can install games on your laptop, but you need to ask the IT team first to ensure it's company-approved a